# Ninai × Google ADK Adapter

Demonstrates `ninai.adapters.adk` — the official Google ADK integration
shipped **inside the Ninai SDK** (`pip install "ninai[adk]"`):

- `get_ninai_adk_tools` — returns search + store as ADK `FunctionTool` instances
- `NinaiADKMemoryService` — persists ADK session events in Ninai

All API calls are **mocked** — runs offline without a live Ninai server.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
SDK_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'sdk', 'python'))
if SDK_PATH not in sys.path:
    sys.path.insert(0, SDK_PATH)

import google.adk
import ninai
from ninai.adapters.adk import (
    get_ninai_adk_tools,
    NinaiADKMemoryService,
)

print('google-adk :', google.adk.__version__)
print('ninai SDK  :', ninai.__version__)
print('Adapter imports OK.')


google-adk : 1.29.0
ninai SDK  : 0.0.1b1
Adapter imports OK.


## Mock Ninai client

In [2]:
from unittest.mock import MagicMock
from types import SimpleNamespace

_STORE: dict = {}
_CTR = [0]

def _mock_create(**kwargs):
    _CTR[0] += 1
    m = SimpleNamespace(
        id=str(_CTR[0]), content=kwargs.get('content', ''),
        title=kwargs.get('title', ''), tags=kwargs.get('tags', []),
    )
    _STORE[m.id] = m
    return m

def _mock_search(query, **kwargs):
    items = [
        SimpleNamespace(memory_id=m.id, content=m.content, score=0.9, title=m.title)
        for m in list(_STORE.values())[-5:]
    ]
    return SimpleNamespace(items=items[:3], total=len(items))

client = MagicMock()
client.memories.create.side_effect = _mock_create
client.memories.search.side_effect = _mock_search
print('Mock client ready')


Mock client ready


## 1. Create ADK tools from Ninai SDK

In [3]:
tools = get_ninai_adk_tools(client)
print(f'ADK tools created: {[t.name for t in tools]}')


ADK tools created: ['ninai_search_memory', 'ninai_store_memory']


## 2. Build an ADK Agent

In [4]:
from google.adk import Agent

agent = Agent(
    name='ninai_enterprise_agent',
    model='gemini-2.0-flash',
    description='Enterprise assistant with Ninai memory',
    instruction=(
        'You are an enterprise assistant. '
        'Use ninai_store_memory to remember important facts. '
        'Use ninai_search_memory to retrieve context before answering.'
    ),
    tools=tools,
)

print(f'Agent : {agent.name}')
print(f'Tools : {[t.name for t in agent.tools]}')


Agent : ninai_enterprise_agent
Tools : ['ninai_search_memory', 'ninai_store_memory']


## 3. Direct tool invocation smoke test

In [5]:
# Direct tool invocations — no live LLM call needed
search_fn, store_fn = tools[0].func, tools[1].func

store_fn('Board approved $10M Series A on 2026-04-01', tags='finance,funding')
store_fn('Engineering headcount target: 25 by Q4', tags='hr,headcount')

hits = search_fn('funding round')
print(f'Search hits: {hits["count"]}')
for r in hits['results']:
    print(f'  - {r}')

print('\nADK + Ninai adapter verified.')


Search hits: 2
  - Board approved $10M Series A on 2026-04-01
  - Engineering headcount target: 25 by Q4

ADK + Ninai adapter verified.


## 4. NinaiADKMemoryService — cross-session recall

In [6]:
svc = NinaiADKMemoryService(ninai_client=client)
svc.save_session_event('sess-99', {'role': 'user',  'content': 'Schedule Q3 planning call'})
svc.save_session_event('sess-99', {'role': 'agent', 'content': 'Q3 planning call scheduled for 2026-06-01'})

hits = svc.search('Q3 planning')
print(f'Recalled {len(hits)} events:')
for h in hits:
    print(f'  {h}')


Recalled 3 events:
  Board approved $10M Series A on 2026-04-01
  Engineering headcount target: 25 by Q4
  Schedule Q3 planning call


## 5. Enterprise feature gating

In [7]:
from ninai.exceptions import EnterpriseFeatureRequired

try:
    raise EnterpriseFeatureRequired(
        'Advanced federated memory requires an Enterprise license.',
        feature='enterprise.federated_memory',
    )
except EnterpriseFeatureRequired as e:
    print(f'Caught: {e}')
    print(f'Feature: {e.feature}')


Caught: Advanced federated memory requires an Enterprise license. (feature=enterprise.federated_memory) — upgrade at https://sansten.com/ninai/pricing
Feature: enterprise.federated_memory
